In [16]:
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch
import os
import json

In [2]:
DATA_DIR = Path.cwd().parent / "data"
ARTIFACTS_DIR = Path.cwd().parent / "artifacts"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
BATCH_SIZE = 128

In [3]:
path = DATA_DIR / "processed" / "preprocessed_medicine_dataset.csv"
df = pd.read_csv(path)
df.head()

,name,Chemical Class,Habit Forming,Therapeutic Class,Action Class,Substitutes,Side Effects,Uses
0,augmentin 625 duo tablet,NaN,No,ANTI INFECTIVES,NaN,"Penciclav 500 mg/125 mg Tablet, Moxikind-CV 62...","Vomiting, Nausea, Diarrhea",Treatment of Bacterial infections
1,azithral 500 tablet,Macrolides,No,ANTI INFECTIVES,Macrolides,"Zithrocare 500mg Tablet, Azax 500 Tablet, Zady...","Vomiting, Nausea, Abdominal pain, Diarrhea",Treatment of Bacterial infections
2,ascoril ls syrup,NaN,No,RESPIRATORY,NaN,"Solvin LS Syrup, Ambrodil-LX Syrup, Zerotuss X...","Nausea, Vomiting, Diarrhea, Upset stomach, Sto...",Treatment of Cough with mucus
3,allegra 120mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Lcfex Tablet, Etofex 120mg Tablet, Nexofex 120...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...
4,avil 25 tablet,Pyridines Derivatives,No,RESPIRATORY,H1 Antihistaminics (First Generation),Eralet 25mg Tablet,"Sleepiness, Dryness in mouth",Treatment of Allergic conditions


In [4]:
print("Dataset shape:", df.shape)

Dataset shape: (248218, 8)


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248218 entries, 0 to 248217
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   name               248218 non-null  object
 1   Chemical Class     137791 non-null  object
 2   Habit Forming      248218 non-null  object
 3   Therapeutic Class  248149 non-null  object
 4   Action Class       138036 non-null  object
 5   Substitutes        238621 non-null  object
 6   Side Effects       248218 non-null  object
 7   Uses               248218 non-null  object
dtypes: object(8)
memory usage: 15.2+ MB


In [6]:
print(df.columns.tolist())

['name', 'Chemical Class', 'Habit Forming', 'Therapeutic Class', 'Action Class', 'Substitutes', 'Side Effects', 'Uses']


In [7]:
TEXT_COLUMNS = ['name', 'Chemical Class', 'Habit Forming', 'Therapeutic Class', 'Action Class', 
                'Substitutes', 'Side Effects', 'Uses']

In [8]:
def create_document(row):
    parts = {}

    for column in TEXT_COLUMNS:
        value = row[column]

        if pd.notna(value):
            value = str(value).strip()

            if value:
                parts[column] = value

    return " ".join([f"{k}: {v}" for k, v in parts.items()])

df["document"] = df.apply(create_document, axis=1)

In [9]:
df.head()

,name,Chemical Class,Habit Forming,Therapeutic Class,Action Class,Substitutes,Side Effects,Uses,document
0,augmentin 625 duo tablet,NaN,No,ANTI INFECTIVES,NaN,"Penciclav 500 mg/125 mg Tablet, Moxikind-CV 62...","Vomiting, Nausea, Diarrhea",Treatment of Bacterial infections,name: augmentin 625 duo tablet Habit Forming: ...
1,azithral 500 tablet,Macrolides,No,ANTI INFECTIVES,Macrolides,"Zithrocare 500mg Tablet, Azax 500 Tablet, Zady...","Vomiting, Nausea, Abdominal pain, Diarrhea",Treatment of Bacterial infections,name: azithral 500 tablet Chemical Class: Macr...
2,ascoril ls syrup,NaN,No,RESPIRATORY,NaN,"Solvin LS Syrup, Ambrodil-LX Syrup, Zerotuss X...","Nausea, Vomiting, Diarrhea, Upset stomach, Sto...",Treatment of Cough with mucus,name: ascoril ls syrup Habit Forming: No Thera...
3,allegra 120mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Lcfex Tablet, Etofex 120mg Tablet, Nexofex 120...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,name: allegra 120mg tablet Chemical Class: Dip...
4,avil 25 tablet,Pyridines Derivatives,No,RESPIRATORY,H1 Antihistaminics (First Generation),Eralet 25mg Tablet,"Sleepiness, Dryness in mouth",Treatment of Allergic conditions,name: avil 25 tablet Chemical Class: Pyridines...


In [10]:
df["document"][0]

'name: augmentin 625 duo tablet Habit Forming: No Therapeutic Class: ANTI INFECTIVES Substitutes: Penciclav 500 mg/125 mg Tablet, Moxikind-CV 625 Tablet, Moxiforce-CV 625 Tablet, Fightox 625 Tablet, Novamox CV 625mg Tablet Side Effects: Vomiting, Nausea, Diarrhea Uses: Treatment of Bacterial infections'

In [11]:
model = SentenceTransformer(EMBEDDING_MODEL, device = "cuda")
print("Model embedding dimension: ", model.get_embedding_dimension())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2722.87it/s]


Model embedding dimension:  384


In [12]:
print(model.device)

cuda:0


In [13]:
documents = df["document"].tolist()

embeddings = model.encode(
    documents,
    show_progress_bar = True,
    batch_size = BATCH_SIZE,
    convert_to_numpy = True,
    normalize_embeddings = True 
)

print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 1940/1940 [11:05<00:00,  2.92it/s]


Embeddings shape: (248218, 384)


In [14]:
print("Shape:", embeddings.shape)
print("dtype:", embeddings.dtype)

print("\nFirst embedding:")
print(embeddings[0][:20])

Shape: (248218, 384)
dtype: float32

First embedding:
[-0.02969329 -0.00560662 -0.01583963  0.02016684  0.04684112 -0.01761435
  0.05088963  0.09216329 -0.05666829  0.00136663  0.00634531 -0.02761245
  0.03655448 -0.01140929  0.04850787 -0.01488204  0.0361762  -0.03567458
 -0.03865918  0.01329539]


In [15]:
norms = np.linalg.norm(embeddings[:100], axis=1)

print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())

Minimum norm: 0.99999994
Maximum norm: 1.0000001


In [17]:
embedding_path = os.path.join(ARTIFACTS_DIR, "embeddings.npy")

np.save(embedding_path, embeddings)

print(f"Saved embeddings to: {embedding_path}")

Saved embeddings to: d:\Projects\MIRx\artifacts\embeddings.npy


In [18]:
metadata = df.drop(columns=["document"]).copy()

metadata["document"] = df["document"]

metadata_path = os.path.join(ARTIFACTS_DIR, "metadata.parquet")

metadata.to_parquet(metadata_path, index=False)

print(f"Saved metadata to: {metadata_path}")

Saved metadata to: d:\Projects\MIRx\artifacts\metadata.parquet


In [19]:
config = {
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": int(model.get_embedding_dimension()),
    "normalized": True,
    "batch_size": BATCH_SIZE,
    "num_documents": len(df),
    "text_columns": TEXT_COLUMNS
}

config_path = os.path.join(ARTIFACTS_DIR, "embedding_config.json")

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"Saved configuration to: {config_path}")

Saved configuration to: d:\Projects\MIRx\artifacts\embedding_config.json


In [20]:
print("EMBEDDING PIPELINE VERDICT")

print("Documents :", len(df))
print("Dimensions:", embeddings.shape[1])
print("Model     :", EMBEDDING_MODEL)
print("Embeddings:", embedding_path)
print("Metadata  :", metadata_path)
print("Config    :", config_path)

EMBEDDING PIPELINE VERDICT
Documents : 248218
Dimensions: 384
Model     : BAAI/bge-small-en-v1.5
Embeddings: d:\Projects\MIRx\artifacts\embeddings.npy
Metadata  : d:\Projects\MIRx\artifacts\metadata.parquet
Config    : d:\Projects\MIRx\artifacts\embedding_config.json
